# 06 · Integrar, verificar e avaliar

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from pprint import pprint
from src.config import DATA

O mesmo `Radar.chat` atende FastAPI, CLI, notebooks e avaliação. O executor
        usa `create_openai_tools_agent` e `AgentExecutor`. A célula abaixo demonstra a
        integração com uma decisão roteirizada, sem credenciais. No modo real, o LLM
        configurado em `OPENAI_MODEL` decide quais ferramentas chamar. O verificador
        combina regras geográficas, NLI e, no modo real, uma checagem de coerência pelo
        LLM antes da redação.

In [2]:
from fastapi.testclient import TestClient
from unittest.mock import patch
from src.agente import Radar
from src.api import app
from src.demo import QUESTIONS
# Cenário reprodutível (vagas fictícias com o caso Atlas), independente do MODO_DUMMY do .env.
patcher = patch('src.api.get_radar', return_value=Radar(dummy=True))
patcher.start()
client = TestClient(app)
print('Interface HTTP:', client.get('/').status_code)
for q in QUESTIONS:
    res = client.post('/chat', json={'session_id': 'notebook-05', 'message': q})
    res.raise_for_status()
    data = res.json()
    print(data['answer'])
    print('Ferramentas:', [item['name'] for item in data['tool_calls']])
    if q == QUESTIONS[0]:
        pprint(next(v for v in data['verification'] if v['job_id'] == 'demo-atlas'))
combined = client.post('/chat', json={'session_id': 'notebook-05-combinado',
    'message': 'Busque vagas de Python e compare com meu currículo'}).json()
print('Pedido combinado:', combined['answer'].splitlines()[-1])
print('Trajetória:', [t['name'] for t in combined['tool_calls']])
patcher.stop()

/var/folders/fp/5_zj8d4n2g1fr35b3rkfs9c40000gn/T/ipykernel_11663/3421822002.py:7: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  patcher = patch('src.api.get_radar', return_value=Radar(dummy=True))
<string>:3: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
tool=buscar_vagas source=Remotive status=dummy


tool=buscar_vagas source=Arbeitnow status=dummy


/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Interface HTTP: 200


W1112 23:07:53.381000 11663 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


... (10 outputs left out)

tool=recuperar_contexto source=FAISS documental status=ok


2 vaga(s) recomendada(s); 1 pendente(s); 2 descartada(s) na amostra exibida.
1. Senior Python Backend Engineer · Aurora Labs.
2. Python Platform Engineer · Orbit Works.
Ferramentas: ['buscar_vagas', 'buscar_vagas']
{'claim': 'Applicants based in Brazil are eligible.',
 'evidence': 'Remote, US only.',
 'job_id': 'demo-atlas',
 'reason': 'O anúncio exclui candidatos no Brasil. NLI: contradiction. '
           'Correção aplicada: vaga excluída da lista.',
 'status': 'contradita'}
2 vaga(s) recomendada(s); 1 pendente(s); 2 descartada(s) na amostra exibida.
1. Senior Python Backend Engineer · Aurora Labs.
2. Python Platform Engineer · Orbit Works.
A lista numerada inclui apenas vagas com elegibilidade confirmada para o Brasil. Os demais casos aparecem como evidência.
Ferramentas: ['buscar_vagas', 'buscar_vagas']
Python Platform Engineer: EUR 5,400.00 por mês ≈ R$ 32,400.00 por mês. Cotação fictícia de 2026-01-01 (1 EUR = R$ 6.00).
Ferramentas: ['converter_moeda']
Senior Python Backend Engin

Pedido combinado: Melhor aderência ao currículo: Senior Python Backend Engineer. Habilidades em comum: Python, FastAPI, PostgreSQL.
Trajetória: ['buscar_vagas', 'buscar_vagas', 'recuperar_contexto']


False

In [3]:
from scripts.evaluate import local_report, upload_experiment
pprint(local_report())
if os.getenv('NOTEBOOK_LANGSMITH') == '1':
    upload_experiment()
else:
    print('Regressão local; esta célula não publica um novo experimento no LangSmith.')

/Users/fernandotrinta/Desktop/Infnet/scripts/evaluate.py:28: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  radar = Radar(db_path=Path(temp) / "vagas.db")
tool=buscar_vagas source=Remotive status=dummy


tool=buscar_vagas source=Arbeitnow status=dummy


tool=buscar_vagas source=Remotive status=dummy


tool=buscar_vagas source=Arbeitnow status=dummy


/Users/fernandotrinta/Desktop/Infnet/scripts/evaluate.py:28: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  radar = Radar(db_path=Path(temp) / "vagas.db")


tool=buscar_vagas source=Remotive status=dummy


... (73 outputs left out)

12it [00:45,  3.76s/it]

/Users/fernandotrinta/Desktop/Infnet/scripts/evaluate.py:130: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  url = client.get_run_url(run=next(client.list_runs(project_name=project.name, is_root=True)))


Experimento: radar-integrador-0b94dc53
URL do experimento: https://smith.langchain.com/o/f3d9bd48-9e7a-4483-9e65-dd3af105169e/projects/p/5688720d-d3a7-426b-8f38-82d5034202ee
Dataset: 96eed1b9-7c17-4bc7-8f03-dffe2d66807b
Execução: https://smith.langchain.com/o/f3d9bd48-9e7a-4483-9e65-dd3af105169e/projects/p/5688720d-d3a7-426b-8f38-82d5034202ee/r/01a1d2a9-712e-7d23-a7a9-04c909cc0905?poll=true



/Users/fernandotrinta/Desktop/Infnet/scripts/evaluate.py:130: LangSmithDeprecationWarning: get_run_url() is deprecated and will be removed after Jan 31, 2027. Use client.runs.get_url() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration-runs#runs-get-url for the migration guide.
  url = client.get_run_url(run=next(client.list_runs(project_name=project.name, is_root=True)))


## Resultados publicados

        A saída da célula anterior é a regressão local. O aviso histórico nela se refere
        àquela execução do notebook. Os experimentos foram publicados depois:
        [demo com 12 casos](../output/langsmith.json) e
        [modelo real com 6 casos](../output/langsmith-real.json). Os registros trazem os
        links do LangSmith; o relatório técnico mostra as telas da suíte real.

        Na suíte real, o mesmo `gpt-6-luna` fez o papel de agente e de juiz em chamadas
        separadas. O código comparou rotas e ferramentas por turno. Os anúncios, a
        cotação e o currículo eram fixos. A taxa de afirmações confirmadas na demo não
        é acurácia: inclui situações em que o resultado correto era contradito ou incerto.

        O item 5.5 pede Streamlit. A interface entregue é Vue com FastAPI; a CLI também
        executa o agente.

## Evidência

- **5.1** → [integracao](#integracao): Verificador registra a contradição do caso Atlas; a suíte real registra o veredito e o rótulo NLI.
- **5.2** → [integracao](#integracao): Busca e RAG no mesmo pedido; câmbio no mesmo Radar.
- **5.3** → [integracao](#integracao): Executor exercitado no modo roteirizado abaixo e com OpenAI na suíte real publicada.
- **5.4** → [integracao](#integracao): Histórico preserva referência à segunda vaga.
- **5.5** → [integracao](#integracao): Streamlit não implementado. Interface Vue com FastAPI e CLI.
- **5.6** → [avaliacao](#avaliacao): Regressão local e experimentos publicados no LangSmith, com qualidade e trajetória.